# Molecular docking -- Mentha_spicata vs PPAR-alpha (1K7L)

Protein: Human PPAR-alpha ligand-binding domain, PDB 1K7L, native agonist GW409544 (ligand code 544). Only chain A (the receptor) is used for docking -- the crystal also contains a short co-activator peptide (chain B) bound to the receptor surface, a real part of the active conformation but not the small-molecule ligand pocket, excluded here as is standard practice for nuclear-receptor docking. Compounds: Carvone, Limonene, cis-Dihydrocarvone, beta-Caryophyllene (as specified by the user; SMILES cross-checked against PubChem canonical structures).

## Content of this notebook

1. Feching system and cleanup
2. Protein and ligand sanitization
3. System Visualization 
4. Docking with AutoDock Vina
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - PDBQT results file conversion to SDF
    - 3D visualization of docking results
    - 2D interaction table and map
5. Docking with Smina
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - 3D visualization of docking results
    - 2D interaction table and map
6. Docking with LeDock
    - Receptor preparation
    - Ligand preparation
    - Docking box definition
    - Docking
    - DOK results file conversion to SDF
    - 3D visualization of docking results
    - 2D interaction table and map

In [ ]:
from pymol import cmd
import py3Dmol

from vina import Vina

from openbabel import pybel

from rdkit import Chem
from rdkit.Chem import AllChem, Draw

from meeko import MoleculePreparation

import MDAnalysis as mda
from MDAnalysis.coordinates import PDB

import prolif as plf
from prolif.plotting.network import LigNetwork


import sys, os
sys.path.insert(1, "/home/ali/medical plants/docking/Jupyter_Dock-main/utilities/")
from utils import fix_protein, getbox, generate_ledock_file, pdbqt_to_sdf, dok_to_sdf


import warnings
warnings.filterwarnings("ignore")
%config Completer.use_jedi = False

It's a good idea to run the test protocols before attempting custom projects. Later, the user can specify the location of his/her project and save all of the files separately.

In [ ]:
os.chdir('/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/')


## 1. Feching system and cleanup

Implementing Pymol is a simple way to download PDB structures. The user can launch this or any other Jupyter Dock's protocol by providing his or her own files.

In [ ]:
cmd.fetch(code='1K7L',type='pdb1')

# 1K7L contains the PPAR-alpha LBD (chain A) PLUS a short co-activator
# peptide (chain B) bound to the receptor surface (part of the real active
# conformation, but not the small-molecule ligand pocket). Keep chain A
# only for the docking receptor.
cmd.select(name='Prot', selection='polymer.protein and chain A')
cmd.save(filename='1K7L_clean.pdb', format='pdb', selection='Prot', state=1)

# Native ligand: GW409544, PDB ligand code 544, chain A.
cmd.select(name='NativeLig', selection='resn 544 and chain A')
_natlig_model = cmd.get_model('NativeLig', state=1)
print("Native ligand (GW409544) atom count:", len(_natlig_model.atom))
cmd.save(filename='1K7L_native_lig.mol2', format='mol2', selection='NativeLig', state=1)
cmd.delete('all')


## 2. Protein and ligand sanitization

### 2.1. Protein Sanitization

#### Method 1: LePro

The Lephar molecular docking suite includes a very powerful tool for automatically preparing proteins for molecular docking. As a result, for these protocols, it will be the preferred tool for protein preparation. The protein prepared by LePro can be used in AutoDock VIna and LeDock.

In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" # Launch this cell to see parameters


In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86" {'1K7L_clean.pdb'}

os.rename('pro.pdb','1K7L_clean_H.pdb')


#### Method 2: fix_protein (PDBFixer)

For proteins with missing amino acids or residues, or to ensure a more thorough sanitization of protein Jupyter Dock includes the _fix_protein()_ function, which employs PDBFixer to correct a wide range of common errors in protein pdb files. Furthermore, PDBFixer enables the assignment of pH-dependent protonation states to proteins.


>**Warning 1:**  It is possible to encounter problems with protein fixing when using **_fix_protein()_** and AutoDock Tools' **_prepare_receptor_** or when running LeDock. To resolve the issue, it is best to set the parameter _-A hydrogens_ in _prepare receptor_.


>**Hint:** PDBFixer is a great solution for many systems because it can solve serious problems in PDB files. As a result, PDBFixer renumbers the residues beginning with 1 regardless of the numbering on the original PDB file. To address this issue, the **_fix_protein()_** function includes a protocol for atomically renumbering the residues in accordance with the original PDB file.


**_fix_protein ( params )_**

Params:
      
  - **filename**: _str or path-like_ ; input file containing protein struture to be modified, file extrension must be pdb

  - **addHs_pH**: _float_ ; Add hydrogens at user defined pH

  - **try_renumberResidues**: _bool_ ; By default PDBFixer renumarets residues starting in 1, this option tries to recover originar residues numbering
    
  - **output**: _str or path-like_ ; output filename, extension must be pdb

```
fix_protein(filename='1K7L_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='1K7L_clean_H.pdb')
```

## 4. Docking with AutoDock Vina

AutoDock Vina (Vina) is one of the docking engines in the AutoDock Suite, together with AutoDock4 (AD4), AutoDockGPU, AutoDockFR, and AutoDock-CrankPep, and arguably among the most widely used and successful docking engines. The reasons for this success are mostly due to its ease of use and its speed (up to 100x faster than AD4), when compared to the other docking engines in the suite and elsewhere, as well as being open source.

### 4.1. Protein preparation

After sanitization, the protein docking preparation includes converting it to the PDBQT file format, which stores the atomic coordinates, partial charges, and AutoDock atom types for both the receptor and the ligand. 

>**Hint:** Despite the fact that the PDBQT format includes changes (Q) and atom types (T) for molecular docking. The charges are not required for Autodock Vina, which computes electrostatic interactions using its own force field. Although, when using AutoDock instead of AutoDock Vina, the Q term is required. More information can be found here:https://autodock-vina.readthedocs.io/en/latest/introduction.html

#### Method 1: AutoDock Tools prepare_receptor

The AutoDock Tools are the best way to prepare the receptor for AutoDock Vina. Nonetheless, AutoDock Tools is a comprehensive suite of programs and scripts that are difficult to manage. Jupyter Dock execute the **_prepare receptor_** and **_prepare_ligand_** functions on their own. As a result, obtaining proper PDBQT files for AutoDock Vina has never been easier.

>**Warning:** There is currently only one method for preparing a PDBQT file for a receptor. It is expected that new methods apart from AutoDock Tools executables will be included in the near future into Jupyter Dock.

In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" #Lauch this cell to see parameters


In [ ]:
!"/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor" -v -r {'1K7L_clean_H.pdb'} -o {'1K7L_clean_H.pdbqt'}

# If this raises "UnboundLocalError: local variable 'bonds' referenced before
# assignment", use this fallback:
# import subprocess
# r = subprocess.run(["obabel", "1K7L_clean_H.pdb", "-O", "1K7L_clean_H.pdbqt", "-xr"],
#                     capture_output=True, text=True)
# print(r.stdout, r.stderr)


In [ ]:
fix_protein(filename='1K7L_clean.pdb',addHs_pH=7.4,try_renumberResidues=True,output='1K7L_clean_H_fix.pdb')

## 7. Screening PHYTO compounds (Mentha_spicata) against PPAR-alpha (1K7L)

Reuses the receptor (`1K7L_clean_H.pdb`/`.pdbqt`, chain A only) and the
docking box (native ligand GW409544, chain A, `state=1`) prepared above.
Each compound docked in its own cell with its own uniquely-named
variables, seed pinned at 42. Compounds: Carvone, Limonene,
cis-Dihydrocarvone, beta-Caryophyllene (as specified by the user).

Reference from the PPAR-alpha literature: the real synthetic PPARalpha
drug fenofibrate scores -6.24 kcal/mol in a comparable docking study.

In [ ]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from openbabel import openbabel as ob
from vina import Vina
import pandas as pd
import matplotlib.pyplot as plt

DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"
FIGURES_DIR = os.path.join(DOCKING_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.chdir(DOCKING_DIR)

RECEPTOR_PDB   = os.path.join(DOCKING_DIR, "1K7L_clean_H.pdb")
RECEPTOR_PDBQT = os.path.join(DOCKING_DIR, "1K7L_clean_H.pdbqt")
NATIVE_LIG     = os.path.join(DOCKING_DIR, "1K7L_native_lig.mol2")

assert os.path.exists(RECEPTOR_PDBQT), "Run the receptor prep cells above first."
assert os.path.exists(NATIVE_LIG), "Run the native ligand extraction cell above first."

VINA_SEED = 42  # pinned for reproducibility

def prepare_ligand_from_smiles(name, smiles, n_confs=20):
    """SMILES -> multiple 3D conformers (ETKDGv3) -> MMFF94 minimize each ->
    keep lowest-energy conformer -> PDB -> PDBQT (AutoDockTools prepare_ligand,
    with OpenBabel fallback)."""
    import subprocess
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    conf_ids = AllChem.EmbedMultipleConfs(m, numConfs=n_confs, params=params)

    energies = []
    for cid in conf_ids:
        ff = AllChem.MMFFGetMoleculeForceField(m, AllChem.MMFFGetMoleculeProperties(m), confId=cid)
        if ff is None:
            continue
        ff.Minimize(maxIts=2000)
        energies.append((ff.CalcEnergy(), cid))
    if not energies:
        for cid in conf_ids:
            ff = AllChem.UFFGetMoleculeForceField(m, confId=cid)
            ff.Minimize(maxIts=2000)
            energies.append((ff.CalcEnergy(), cid))

    best_energy, best_cid = min(energies, key=lambda x: x[0])
    print(f"  {name}: {len(conf_ids)} conformers generated, "
          f"lowest energy = {best_energy:.2f} kcal/mol (conf {best_cid})")

    pdb_path   = os.path.join(DOCKING_DIR, f'_lig_{name}.pdb')
    pdbqt_path = os.path.join(DOCKING_DIR, f'_lig_{name}.pdbqt')
    Chem.MolToPDBFile(m, pdb_path, confId=best_cid)

    PREP_LIG_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand"
    result = subprocess.run([PREP_LIG_BIN, "-v", "-l", pdb_path, "-o", pdbqt_path],
                             capture_output=True, text=True)
    if not os.path.exists(pdbqt_path) or os.path.getsize(pdbqt_path) == 0:
        obConv = ob.OBConversion()
        obConv.SetInAndOutFormats('pdb', 'pdbqt')
        obmol = ob.OBMol()
        obConv.ReadFile(obmol, pdb_path)
        obConv.WriteFile(obmol, pdbqt_path)

    return pdbqt_path


### 7.1. Recompute the docking box (native ligand GW409544, `state=1`)

In [ ]:
cmd.load(filename=RECEPTOR_PDB, format='pdb', object='prot', state=1)
cmd.load(filename=NATIVE_LIG, format='mol2', object='natlig', state=1)

center, size = getbox(selection='natlig', extending=5.0, software='vina')
cmd.delete('all')

print(center, '\n', size)


In [ ]:
from pymol import cmd
from pymol.cgo import LINEWIDTH, BEGIN, LINES, VERTEX, END, COLOR

cmd.reinitialize()
cmd.load(RECEPTOR_PDB, 'receptor')
cmd.load(NATIVE_LIG, 'native_lig')

cmd.hide('everything')
cmd.show('cartoon', 'receptor')
cmd.color('white', 'receptor')
cmd.show('sticks', 'native_lig')
cmd.color('magenta', 'native_lig')

# Box half-widths
cx, cy, cz = center['center_x'], center['center_y'], center['center_z']
hx, hy, hz = size['size_x']/2, size['size_y']/2, size['size_z']/2

# 8 corners of the box
corners = [
    (cx-hx, cy-hy, cz-hz), (cx+hx, cy-hy, cz-hz),
    (cx+hx, cy+hy, cz-hz), (cx-hx, cy+hy, cz-hz),
    (cx-hx, cy-hy, cz+hz), (cx+hx, cy-hy, cz+hz),
    (cx+hx, cy+hy, cz+hz), (cx-hx, cy+hy, cz+hz),
]

# 12 edges of the box, as pairs of corner indices
edges = [
    (0,1),(1,2),(2,3),(3,0),   # bottom face
    (4,5),(5,6),(6,7),(7,4),   # top face
    (0,4),(1,5),(2,6),(3,7),   # verticals
]

box_cgo = [LINEWIDTH, 3.0, COLOR, 1.0, 1.0, 0.0, BEGIN, LINES]
for a, b in edges:
    box_cgo.extend([VERTEX, *corners[a]])
    box_cgo.extend([VERTEX, *corners[b]])
box_cgo.append(END)

cmd.load_cgo(box_cgo, 'grid_box')

cmd.zoom('receptor')
cmd.bg_color('white')
cmd.set('ray_opaque_background', 1)
cmd.png('1EVE_docking_grid.png', width=1200, height=1200, dpi=300, ray=1)

print("Saved: 1K7L_docking_grid.png")

### 7.2. Dock **Carvone** against PPAR-alpha (1K7L)

In [ ]:
# ---- Carvone (Mentha_spicata, Bardaweel et al. 2018) ----
# Converted from AutoDock Vina to Smina: Vina's redocking validation for
# PPAR-alpha failed (RMSD 2.728 A at exhaustiveness=72); Smina's redocking
# of the same system PASSED (RMSD 1.539 A). Smina is used here accordingly.
import subprocess
SMINA_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/smina"

smiles_Carvone = "CC1=CCC(CC1=O)C(=C)C"
lig_pdbqt_Carvone = prepare_ligand_from_smiles("Carvone", smiles_Carvone)

smina_out_sdf_Carvone = os.path.join(DOCKING_DIR, "Carvone_1K7L_smina_out.sdf")

r_Carvone = subprocess.run([
    SMINA_BIN, "-r", RECEPTOR_PDB, "-l", lig_pdbqt_Carvone,
    "-o", smina_out_sdf_Carvone,
    "--center_x", str(center['center_x']), "--center_y", str(center['center_y']), "--center_z", str(center['center_z']),
    "--size_x", str(size['size_x']), "--size_y", str(size['size_y']), "--size_z", str(size['size_z']),
    "--exhaustiveness", "10", "--num_modes", "10",
], capture_output=True, text=True)
print(r_Carvone.stdout[-800:])
print(r_Carvone.stderr[-400:])
assert os.path.exists(smina_out_sdf_Carvone), "Smina did not produce an output file -- see stdout/stderr above."

smina_poses_Carvone = Chem.SDMolSupplier(smina_out_sdf_Carvone, removeHs=True, sanitize=False)
best_score_Carvone = None
print("=== Carvone (Smina) ===")
for i, pose in enumerate(smina_poses_Carvone):
    if pose is None:
        continue
    score = float(pose.GetProp("minimizedAffinity"))
    if best_score_Carvone is None:
        best_score_Carvone = round(score, 3)
    print(f"  Pose {i+1}: {score:.3f}")
print(f"Best Smina score: {best_score_Carvone} kcal/mol")


### 7.3. Dock **Limonene** against PPAR-alpha (1K7L)

In [ ]:
# ---- Limonene (Mentha_spicata, Bardaweel et al. 2018) ----
# Converted from AutoDock Vina to Smina: Vina's redocking validation for
# PPAR-alpha failed (RMSD 2.728 A at exhaustiveness=72); Smina's redocking
# of the same system PASSED (RMSD 1.539 A). Smina is used here accordingly.
import subprocess
SMINA_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/smina"

smiles_Limonene = "CC1=CCC(CC1)C(=C)C"
lig_pdbqt_Limonene = prepare_ligand_from_smiles("Limonene", smiles_Limonene)

smina_out_sdf_Limonene = os.path.join(DOCKING_DIR, "Limonene_1K7L_smina_out.sdf")

r_Limonene = subprocess.run([
    SMINA_BIN, "-r", RECEPTOR_PDB, "-l", lig_pdbqt_Limonene,
    "-o", smina_out_sdf_Limonene,
    "--center_x", str(center['center_x']), "--center_y", str(center['center_y']), "--center_z", str(center['center_z']),
    "--size_x", str(size['size_x']), "--size_y", str(size['size_y']), "--size_z", str(size['size_z']),
    "--exhaustiveness", "10", "--num_modes", "10",
], capture_output=True, text=True)
print(r_Limonene.stdout[-800:])
print(r_Limonene.stderr[-400:])
assert os.path.exists(smina_out_sdf_Limonene), "Smina did not produce an output file -- see stdout/stderr above."

smina_poses_Limonene = Chem.SDMolSupplier(smina_out_sdf_Limonene, removeHs=True, sanitize=False)
best_score_Limonene = None
print("=== Limonene (Smina) ===")
for i, pose in enumerate(smina_poses_Limonene):
    if pose is None:
        continue
    score = float(pose.GetProp("minimizedAffinity"))
    if best_score_Limonene is None:
        best_score_Limonene = round(score, 3)
    print(f"  Pose {i+1}: {score:.3f}")
print(f"Best Smina score: {best_score_Limonene} kcal/mol")


### 7.4. Dock **cis-Dihydrocarvone** against PPAR-alpha (1K7L)

In [ ]:
# ---- Dihydrocarvone (Mentha_spicata, Bardaweel et al. 2018) ----
# Converted from AutoDock Vina to Smina: Vina's redocking validation for
# PPAR-alpha failed (RMSD 2.728 A at exhaustiveness=72); Smina's redocking
# of the same system PASSED (RMSD 1.539 A). Smina is used here accordingly.
import subprocess
SMINA_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/smina"

smiles_Dihydrocarvone = "C[C@@H]1CC[C@@H](CC1=O)C(=C)C"
lig_pdbqt_Dihydrocarvone = prepare_ligand_from_smiles("Dihydrocarvone", smiles_Dihydrocarvone)

smina_out_sdf_Dihydrocarvone = os.path.join(DOCKING_DIR, "Dihydrocarvone_1K7L_smina_out.sdf")

r_Dihydrocarvone = subprocess.run([
    SMINA_BIN, "-r", RECEPTOR_PDB, "-l", lig_pdbqt_Dihydrocarvone,
    "-o", smina_out_sdf_Dihydrocarvone,
    "--center_x", str(center['center_x']), "--center_y", str(center['center_y']), "--center_z", str(center['center_z']),
    "--size_x", str(size['size_x']), "--size_y", str(size['size_y']), "--size_z", str(size['size_z']),
    "--exhaustiveness", "10", "--num_modes", "10",
], capture_output=True, text=True)
print(r_Dihydrocarvone.stdout[-800:])
print(r_Dihydrocarvone.stderr[-400:])
assert os.path.exists(smina_out_sdf_Dihydrocarvone), "Smina did not produce an output file -- see stdout/stderr above."

smina_poses_Dihydrocarvone = Chem.SDMolSupplier(smina_out_sdf_Dihydrocarvone, removeHs=True, sanitize=False)
best_score_Dihydrocarvone = None
print("=== Dihydrocarvone (Smina) ===")
for i, pose in enumerate(smina_poses_Dihydrocarvone):
    if pose is None:
        continue
    score = float(pose.GetProp("minimizedAffinity"))
    if best_score_Dihydrocarvone is None:
        best_score_Dihydrocarvone = round(score, 3)
    print(f"  Pose {i+1}: {score:.3f}")
print(f"Best Smina score: {best_score_Dihydrocarvone} kcal/mol")


### 7.5. Dock **beta-Caryophyllene** against PPAR-alpha (1K7L)

In [ ]:
# ---- Caryophyllene (Mentha_spicata, Bardaweel et al. 2018) ----
# Converted from AutoDock Vina to Smina: Vina's redocking validation for
# PPAR-alpha failed (RMSD 2.728 A at exhaustiveness=72); Smina's redocking
# of the same system PASSED (RMSD 1.539 A). Smina is used here accordingly.
import subprocess
SMINA_BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/smina"

smiles_Caryophyllene = "CC1=CCCC(=C)C2CC(C2CC1)(C)C"
lig_pdbqt_Caryophyllene = prepare_ligand_from_smiles("Caryophyllene", smiles_Caryophyllene)

smina_out_sdf_Caryophyllene = os.path.join(DOCKING_DIR, "Caryophyllene_1K7L_smina_out.sdf")

r_Caryophyllene = subprocess.run([
    SMINA_BIN, "-r", RECEPTOR_PDB, "-l", lig_pdbqt_Caryophyllene,
    "-o", smina_out_sdf_Caryophyllene,
    "--center_x", str(center['center_x']), "--center_y", str(center['center_y']), "--center_z", str(center['center_z']),
    "--size_x", str(size['size_x']), "--size_y", str(size['size_y']), "--size_z", str(size['size_z']),
    "--exhaustiveness", "10", "--num_modes", "10",
], capture_output=True, text=True)
print(r_Caryophyllene.stdout[-800:])
print(r_Caryophyllene.stderr[-400:])
assert os.path.exists(smina_out_sdf_Caryophyllene), "Smina did not produce an output file -- see stdout/stderr above."

smina_poses_Caryophyllene = Chem.SDMolSupplier(smina_out_sdf_Caryophyllene, removeHs=True, sanitize=False)
best_score_Caryophyllene = None
print("=== Caryophyllene (Smina) ===")
for i, pose in enumerate(smina_poses_Caryophyllene):
    if pose is None:
        continue
    score = float(pose.GetProp("minimizedAffinity"))
    if best_score_Caryophyllene is None:
        best_score_Caryophyllene = round(score, 3)
    print(f"  Pose {i+1}: {score:.3f}")
print(f"Best Smina score: {best_score_Caryophyllene} kcal/mol")


### 7.6. Total docking results table -- Mentha_spicata vs PPAR-alpha (1K7L)

In [ ]:
docking_rows_spicata_PPAR = [
    {"Compound": "Carvone",             "Target": "PPAR-alpha (1K7L)", "Best_Smina_Score_kcal_mol": best_score_Carvone},
    {"Compound": "Limonene",            "Target": "PPAR-alpha (1K7L)", "Best_Smina_Score_kcal_mol": best_score_Limonene},
    {"Compound": "cis-Dihydrocarvone",  "Target": "PPAR-alpha (1K7L)", "Best_Smina_Score_kcal_mol": best_score_Dihydrocarvone},
    {"Compound": "beta-Caryophyllene",  "Target": "PPAR-alpha (1K7L)", "Best_Smina_Score_kcal_mol": best_score_Caryophyllene},
]

docking_df_spicata_PPAR = pd.DataFrame(docking_rows_spicata_PPAR).sort_values("Best_Smina_Score_kcal_mol").reset_index(drop=True)
docking_df_spicata_PPAR.to_csv(os.path.join(DOCKING_DIR, "mentha_spicata_vs_PPAR_smina.csv"), index=False)
docking_df_spicata_PPAR


### 7.7. Figure -- docking score comparison, Mentha_spicata vs PPAR-alpha

### 7.8. ProLif interaction fingerprints -- Mentha_spicata vs PPAR-alpha (each compound, own cell)

`n_jobs=1` avoids a multiprocessing/pickling crash in this environment;
`to_dataframe()` takes no arguments in ProLif 2.2.1. The 2D network is
saved as an interactive HTML file via `LigNetwork.save()` -- the original
notebook's `save_png(path)` call is not valid for this ProLif version
(`save_png()` takes no filename argument), so that call is replaced here.

In [ ]:
prot_fp = mda.Universe("1K7L_clean_H_fix.pdb")  # fix_protein() output
prot_fp = plf.Molecule.from_mda(prot_fp)


In [ ]:
# ---- Helper: render a ProLIF LigNetwork widget to a real PNG file ----
import os
import sys
import subprocess

_RENDER_WORKER_PATH = os.path.join(os.getcwd(), "_lignetwork_render_worker.py")

_RENDER_WORKER_SOURCE = '''
import sys
from playwright.sync_api import sync_playwright

html_path, png_path, width, height, wait_ms = sys.argv[1:6]
width, height, wait_ms = int(width), int(height), int(wait_ms)

with sync_playwright() as p:
    browser = p.chromium.launch()
    page = browser.new_page(viewport={"width": width, "height": height})
    page.goto(f"file://{html_path}")
    page.wait_for_timeout(wait_ms)
    try:
        page.evaluate("if (typeof network !== 'undefined') { network.fit({animation:false}); }")
        page.wait_for_timeout(300)
    except Exception:
        pass
    page.screenshot(path=png_path)
    browser.close()
'''

def lignetwork_to_png(net, png_path, width=1000, height=800, wait_ms=1500):
    """Render a ProLIF LigNetwork object to a standalone PNG file."""
    with open(_RENDER_WORKER_PATH, "w") as f:
        f.write(_RENDER_WORKER_SOURCE)

    tmp_html = png_path.rsplit(".", 1)[0] + "_tmp.html"
    net.save(tmp_html)

    result = subprocess.run(
        [sys.executable, _RENDER_WORKER_PATH, os.path.abspath(tmp_html),
         os.path.abspath(png_path), str(width), str(height), str(wait_ms)],
        capture_output=True, text=True,
    )

    if result.returncode != 0:
        print("Render worker STDOUT:", result.stdout)
        print("Render worker STDERR:", result.stderr)
        raise RuntimeError(f"lignetwork_to_png failed for {png_path}")

    os.remove(tmp_html)
    print("Saved PNG:", png_path)

In [ ]:
# ---- ProLif: Carvone ----
lig_suppl_Carvone = list(plf.sdf_supplier(smina_out_sdf_Carvone))
fp_Carvone = plf.Fingerprint()
fp_Carvone.run_from_iterable(lig_suppl_Carvone, prot_fp, n_jobs=1)
results_df_Carvone = fp_Carvone.to_dataframe()
print("=== Carvone ===")
display(results_df_Carvone)

valid_frames = results_df_Carvone[results_df_Carvone.any(axis=1)].index

if len(valid_frames) == 0:
    print("No interactions detected in any pose for Carvone - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Carvone = fp_Carvone.plot_lignetwork(lig_suppl_Carvone[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Carvone = os.path.join(FIGURES_DIR, "Carvone_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_Carvone, net_png_Carvone)

In [ ]:
# ---- ProLif: Limonene ----
lig_suppl_Limonene = list(plf.sdf_supplier(smina_out_sdf_Limonene))
fp_Limonene = plf.Fingerprint()
fp_Limonene.run_from_iterable(lig_suppl_Limonene, prot_fp, n_jobs=1)
results_df_Limonene = fp_Limonene.to_dataframe()
print("=== Limonene ===")
display(results_df_Limonene)

valid_frames = results_df_Limonene[results_df_Limonene.any(axis=1)].index

if len(valid_frames) == 0:
    print("No interactions detected in any pose for Limonene - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Limonene = fp_Limonene.plot_lignetwork(lig_suppl_Limonene[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Limonene = os.path.join(FIGURES_DIR, "Limonene_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_Limonene, net_png_Limonene)

In [ ]:
# ---- ProLif: cis-Dihydrocarvone ----
lig_suppl_Dihydrocarvone = list(plf.sdf_supplier(smina_out_sdf_Dihydrocarvone))
fp_Dihydrocarvone = plf.Fingerprint()
fp_Dihydrocarvone.run_from_iterable(lig_suppl_Dihydrocarvone, prot_fp, n_jobs=1)
results_df_Dihydrocarvone = fp_Dihydrocarvone.to_dataframe()
print("=== cis-Dihydrocarvone ===")
display(results_df_Dihydrocarvone)

valid_frames = results_df_Dihydrocarvone[results_df_Dihydrocarvone.any(axis=1)].index

if len(valid_frames) == 0:
    print("No interactions detected in any pose for cis-Dihydrocarvone - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Dihydrocarvone = fp_Dihydrocarvone.plot_lignetwork(lig_suppl_Dihydrocarvone[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Dihydrocarvone = os.path.join(FIGURES_DIR, "Dihydrocarvone_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_Dihydrocarvone, net_png_Dihydrocarvone)

In [ ]:
# ---- ProLif: beta-Caryophyllene ----
lig_suppl_Caryophyllene = list(plf.sdf_supplier(smina_out_sdf_Caryophyllene))
fp_Caryophyllene = plf.Fingerprint()
fp_Caryophyllene.run_from_iterable(lig_suppl_Caryophyllene, prot_fp, n_jobs=1)
results_df_Caryophyllene = fp_Caryophyllene.to_dataframe()
print("=== beta-Caryophyllene ===")
display(results_df_Caryophyllene)

valid_frames = results_df_Caryophyllene[results_df_Caryophyllene.any(axis=1)].index

if len(valid_frames) == 0:
    print("No interactions detected in any pose for beta-Caryophyllene - skipping network plot.")
else:
    frame_to_use = valid_frames[0]
    print("Using frame:", frame_to_use)
    net_Caryophyllene = fp_Caryophyllene.plot_lignetwork(lig_suppl_Caryophyllene[0], kind="frame", frame=frame_to_use, rotation=270)
    net_png_Caryophyllene = os.path.join(FIGURES_DIR, "Caryophyllene_1K7L_2Dnetwork.png")
    lignetwork_to_png(net_Caryophyllene, net_png_Caryophyllene)

### 7.9. Figure -- best-scoring compound vs. the real native ligand (GW409544)

In [ ]:
best_row_spicata_PPAR = docking_df_spicata_PPAR.iloc[0]
best_compound_spicata_PPAR = best_row_spicata_PPAR['Compound']
print("Best-scoring compound:", best_compound_spicata_PPAR, "|", best_row_spicata_PPAR['Best_Vina_Score_kcal_mol'], "kcal/mol")

best_sdf_spicata_PPAR = {
    "Carvone":            vina_out_sdf_Carvone,
    "Limonene":           vina_out_sdf_Limonene,
    "cis-Dihydrocarvone": vina_out_sdf_Dihydrocarvone,
    "beta-Caryophyllene": vina_out_sdf_Caryophyllene,
}[best_compound_spicata_PPAR]

view = py3Dmol.view()
view.removeAllModels()
view.setViewStyle({'style':'outline','color':'black','width':0.1})

view.addModel(open(RECEPTOR_PDB,'r').read(), format='pdb')
Prot = view.getModel()
Prot.setStyle({'cartoon':{'arrows':True, 'tubes':True, 'style':'oval', 'color':'white'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.6,'color':'white'})

view.addModel(open(NATIVE_LIG,'r').read(), format='mol2')
ref_m = view.getModel()
ref_m.setStyle({}, {'stick':{'colorscheme':'magentaCarbon','radius':0.2}})

results_spi_PPAR = Chem.SDMolSupplier(best_sdf_spicata_PPAR)
p_spi_PPAR = Chem.MolToMolBlock(results_spi_PPAR[0], False)
print('Reference (GW409544): Magenta | Best compound pose: Cyan')
print('Pose: {} | Score: {}'.format(results_spi_PPAR[0].GetProp('Pose'), results_spi_PPAR[0].GetProp('Score')))

view.addModel(p_spi_PPAR, 'mol')
x = view.getModel()
x.setStyle({}, {'stick':{'colorscheme':'cyanCarbon','radius':0.2}})

view.zoomTo()
view.show()
